---
### 🐛 Found a bug or have a suggestion?
[Open a GitHub Issue](https://github.com/ladataanalytics/pattern-recognition-notebooks/issues/new?template=bug_report.md) — include notebook name, cell number, and what went wrong.
*Search [existing issues](https://github.com/ladataanalytics/pattern-recognition-notebooks/issues) first.*

### 💾 Save your own copy
> **File → Save a copy in Drive** — saves a personal editable copy to your Google Drive.
> Do this before making edits, otherwise changes are lost when the session ends.

# 🎛️ Regularisation — Ridge & Lasso
**ISLP Chapter 6 · Pattern Recognition for the Rest of Us**

> OLS minimises RSS. Ridge and Lasso add a penalty term that shrinks coefficients — trading a little bias for a large reduction in variance. The result: models that generalise far better when predictors are many or correlated.

### What you'll learn
- Why OLS overfits with many correlated predictors
- Ridge regression — L2 penalty, keeps all features
- Lasso regression — L1 penalty, performs automatic variable selection
- Choosing λ with cross-validation
- Comparing OLS, Ridge, and Lasso on real data

### Time: ~50 minutes

In [ ]:
# ⚠️  Run this cell first — sets up data and imports
# Tip: Runtime → Run all  (runs everything top to bottom)

import os, sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

from sklearn.linear_model import Ridge, Lasso, RidgeCV, LassoCV, LinearRegression
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import cross_val_score, train_test_split
from sklearn.metrics import mean_squared_error
import statsmodels.api as sm

plt.rcParams.update({
    'figure.facecolor': 'white', 'axes.facecolor': '#f8f9fa',
    'axes.grid': True, 'grid.alpha': 0.4,
    'axes.spines.top': False, 'axes.spines.right': False,
    'font.size': 11
})

# ── Load Hitters dataset (ISLP Chapter 6) ────────────────────────────────────
try:
    hitters = pd.read_csv('https://www.statlearning.com/s/Hitters.csv')
    print(f'✓ Hitters.csv (ISLP): {hitters.shape}')
except Exception:
    hitters = pd.read_csv('https://raw.githubusercontent.com/ladataanalytics/pattern-recognition-notebooks/main/data/Hitters.csv')
    print(f'✓ Hitters.csv (fallback): {hitters.shape}')

hitters = hitters.dropna()
print(f'  After dropna: {hitters.shape}')

# ── Prepare feature matrix ────────────────────────────────────────────────────
# Encode categorical columns; log-transform Salary (right-skewed)
hitters_enc = pd.get_dummies(hitters, columns=['League','Division','NewLeague'],
                              drop_first=True, dtype=float)
hitters_enc['log_salary'] = np.log(hitters_enc['Salary'])
hitters_enc = hitters_enc.drop(columns=['Salary'])

target   = 'log_salary'
features = [c for c in hitters_enc.columns if c != target]
X = hitters_enc[features].values.astype(float)
y = hitters_enc[target].values

# Train / test split
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.3, random_state=42)

# Scale — Ridge and Lasso require standardised features
scaler  = StandardScaler().fit(X_tr)
X_tr_s  = scaler.transform(X_tr)
X_te_s  = scaler.transform(X_te)

print(f'  Features ({len(features)}): {features}')
print(f'  Train: {X_tr_s.shape}  Test: {X_te_s.shape}')
print(f'  Target: log(Salary) — mean={y.mean():.3f}, std={y.std():.3f}')
print(f'  Python {sys.version.split()[0]} | pandas {pd.__version__}')
print('✓ Setup complete')


## 📐 Part 1 — Why Regularisation?

OLS finds β by minimising **RSS = Σ(yᵢ − Xᵢβ)²**.

With p correlated predictors, small data changes cause wildly different coefficients — **high variance**. Adding a penalty term shrinks coefficients toward zero:

| Method | Objective | Effect |
|--------|-----------|--------|
| **OLS** | Minimise RSS | No constraint — can overfit |
| **Ridge** | Minimise RSS + λΣβⱼ² | Shrinks all coefficients, keeps all features |
| **Lasso** | Minimise RSS + λΣ\|βⱼ\| | Shrinks some coefficients to **exactly zero** — feature selection |

λ is the regularisation strength. λ=0 → OLS. λ→∞ → all coefficients zero.

In [ ]:
# ── Coefficient paths: how Ridge and Lasso shrink as λ grows ─────────────────
alphas = np.logspace(-3, 5, 200)

# Ridge paths
ridge_coefs = []
for a in alphas:
    ridge_coefs.append(Ridge(alpha=a).fit(X_tr_s, y_tr).coef_)
ridge_coefs = np.array(ridge_coefs)

# Lasso paths
lasso_coefs = []
for a in alphas:
    try:
        lasso_coefs.append(
            Lasso(alpha=a, max_iter=10000).fit(X_tr_s, y_tr).coef_)
    except Exception:
        lasso_coefs.append(np.zeros(len(features)))
lasso_coefs = np.array(lasso_coefs)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, coefs, title in zip(axes,
        [ridge_coefs, lasso_coefs],
        ['Ridge — L2 penalty (coefficients shrink but stay non-zero)',
         'Lasso — L1 penalty (coefficients hit zero = feature selection)']):
    for j in range(coefs.shape[1]):
        ax.plot(np.log10(alphas), coefs[:, j], lw=1.2, alpha=0.8)
    ax.axvline(0, color='grey', lw=0.8, ls=':')
    ax.set_xlabel('log₁₀(λ)')
    ax.set_ylabel('Coefficient value')
    ax.set_title(title)
    # Label a few features at left edge
    for j, fname in enumerate(features[:5]):
        ax.annotate(fname, xy=(np.log10(alphas[0]), coefs[0, j]),
                    fontsize=7, alpha=0.7)

plt.suptitle('Regularisation Paths — Hitters Dataset', fontsize=12)
plt.tight_layout()
plt.show()

print("📌 Ridge: all coefficients shrink smoothly — none reach exactly zero.")
print("   Lasso: coefficients hit zero as λ increases — automatic feature selection.")
print("   The point where a coefficient reaches zero = that feature is dropped.")


## 🔬 Part 2 — Ridge Regression

Ridge adds an **L2 penalty**: minimise RSS + λΣβⱼ²

The closed-form solution is:  **β̂_ridge = (XᵀX + λI)⁻¹Xᵀy**

Adding λI makes the matrix invertible even when XᵀX is near-singular (multicollinearity). This is Ridge's key advantage — it works when OLS is numerically unstable.

**Interpretation:** each coefficient is shrunk toward zero by a factor that depends on λ and the eigenvalues of XᵀX.

In [ ]:
# ── Choose optimal λ for Ridge via cross-validation ──────────────────────────
alphas_cv = np.logspace(-3, 5, 200)
ridge_cv  = RidgeCV(alphas=alphas_cv, cv=10).fit(X_tr_s, y_tr)
best_alpha_ridge = ridge_cv.alpha_

print(f"Ridge CV — optimal λ = {best_alpha_ridge:.4f}")
print(f"  (searched {len(alphas_cv)} values from {alphas_cv[0]:.1e} to {alphas_cv[-1]:.1e})")

# Fit Ridge and OLS for comparison
ridge_best = Ridge(alpha=best_alpha_ridge).fit(X_tr_s, y_tr)
ols        = LinearRegression().fit(X_tr_s, y_tr)

# Test MSE
ridge_mse = mean_squared_error(y_te, ridge_best.predict(X_te_s))
ols_mse   = mean_squared_error(y_te, ols.predict(X_te_s))

print(f"\nTest MSE:  OLS = {ols_mse:.4f}  |  Ridge (λ={best_alpha_ridge:.3f}) = {ridge_mse:.4f}")
print(f"Test RMSE: OLS = {ols_mse**0.5:.4f}  |  Ridge = {ridge_mse**0.5:.4f}")
print(f"RMSE ($):  OLS = ${np.exp(ols_mse**0.5)-1:.1%}  |  Ridge = ${np.exp(ridge_mse**0.5)-1:.1%} salary error")

# Coefficient comparison
fig, ax = plt.subplots(figsize=(10, 5))
x = np.arange(len(features))
w = 0.35
ax.bar(x - w/2, ols.coef_,        w, label='OLS',   color='#1e5fa8', alpha=0.8)
ax.bar(x + w/2, ridge_best.coef_, w, label='Ridge',  color='#e85d20', alpha=0.8)
ax.axhline(0, color='grey', lw=0.8)
ax.set_xticks(x); ax.set_xticklabels(features, rotation=45, ha='right', fontsize=8)
ax.set_ylabel('Coefficient value (standardised X)')
ax.set_title(f'OLS vs Ridge (λ={best_alpha_ridge:.3f}) — Hitters Dataset\n'
              'Ridge shrinks all coefficients toward zero')
ax.legend()
plt.tight_layout()
plt.show()

# OLS via statsmodels for p-values on the full model
ols_sm = sm.OLS(y_tr, sm.add_constant(X_tr_s)).fit()
print(f"\nOLS Adj R² = {ols_sm.rsquared_adj:.4f}")
print(f"Significant predictors (p < 0.05): "
      f"{[features[j-1] for j,p in enumerate(ols_sm.pvalues[1:],1) if p < 0.05]}")
print()
print("📌 Ridge coefficients are smaller than OLS — the L2 penalty pulled them in.")
print("   Ridge CANNOT zero out a coefficient — use Lasso for feature selection.")


## 📊 Part 3 — Lasso Regression

Lasso adds an **L1 penalty**: minimise RSS + λΣ|βⱼ|

The L1 geometry creates **corners** at the axes — the RSS ellipse touches a corner before any interior face, driving individual coefficients to **exactly zero**. This is why Lasso does feature selection and Ridge does not.

**Sparse solutions** are useful when:
- p is large and only a few features matter
- You need an interpretable model
- You suspect many predictors are irrelevant noise

In [ ]:
# ── Choose optimal λ for Lasso via cross-validation ──────────────────────────
lasso_cv = LassoCV(alphas=np.logspace(-4, 1, 200), cv=10,
                   max_iter=20000, random_state=42).fit(X_tr_s, y_tr)
best_alpha_lasso = lasso_cv.alpha_

lasso_best    = Lasso(alpha=best_alpha_lasso, max_iter=20000).fit(X_tr_s, y_tr)
n_nonzero     = np.sum(lasso_best.coef_ != 0)
lasso_mse     = mean_squared_error(y_te, lasso_best.predict(X_te_s))

print(f"Lasso CV — optimal λ = {best_alpha_lasso:.4f}")
print(f"  Non-zero coefficients: {n_nonzero} / {len(features)}")
print(f"  Zeroed out: {[features[j] for j,c in enumerate(lasso_best.coef_) if c==0]}")
print(f"\nTest MSE:  Lasso (λ={best_alpha_lasso:.4f}) = {lasso_mse:.4f}")

# Side-by-side: OLS, Ridge, Lasso coefficients
coef_df = pd.DataFrame({
    'Feature': features,
    'OLS':     ols.coef_,
    'Ridge':   ridge_best.coef_,
    'Lasso':   lasso_best.coef_
}).set_index('Feature')

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: all three methods side by side
x  = np.arange(len(features))
w  = 0.26
axes[0].bar(x - w, coef_df['OLS'],   w, label='OLS',   color='#1e5fa8', alpha=0.85)
axes[0].bar(x,     coef_df['Ridge'], w, label='Ridge',  color='#e85d20', alpha=0.85)
axes[0].bar(x + w, coef_df['Lasso'], w, label='Lasso',  color='#1a7a45', alpha=0.85)
axes[0].axhline(0, color='grey', lw=0.8)
axes[0].set_xticks(x)
axes[0].set_xticklabels(features, rotation=45, ha='right', fontsize=8)
axes[0].set_ylabel('Coefficient (standardised X)')
axes[0].set_title('OLS vs Ridge vs Lasso — All Coefficients')
axes[0].legend()

# Right: Lasso only — highlight zeros
colors_lasso = ['#1a7a45' if c != 0 else '#e85d20'
                for c in lasso_best.coef_]
axes[1].bar(x, lasso_best.coef_, color=colors_lasso, alpha=0.85)
axes[1].axhline(0, color='grey', lw=0.8)
axes[1].set_xticks(x)
axes[1].set_xticklabels(features, rotation=45, ha='right', fontsize=8)
axes[1].set_ylabel('Lasso coefficient')
axes[1].set_title(f'Lasso (λ={best_alpha_lasso:.4f}) — Feature Selection\n'
                   f'Green = kept ({n_nonzero}), Red = zeroed ({len(features)-n_nonzero})')

plt.tight_layout()
plt.show()

# Final summary table
print("\n=== Method Comparison ===")
print(f"{'Method':25s}  {'Test MSE':>10}  {'Test RMSE':>10}  {'Features used':>14}")
print("-" * 65)
for name, mse, n_feat in [
        ('OLS',   ols_mse,   len(features)),
        (f'Ridge (λ={best_alpha_ridge:.3f})', ridge_mse, len(features)),
        (f'Lasso (λ={best_alpha_lasso:.4f})', lasso_mse, n_nonzero)]:
    print(f"{name:25s}  {mse:>10.4f}  {mse**0.5:>10.4f}  {n_feat:>14}")

print()
print("📌 Lasso selects a sparse subset — useful when many predictors are noise.")
print("   Ridge keeps all features but reduces their magnitude.")
print("   Neither is always better — use cross-validation to choose.")


## ✅ Part 4 — The Bias-Variance Tradeoff in Regularisation

As λ increases:
- **Bias increases** — we're forcing coefficients away from the OLS estimates
- **Variance decreases** — the model is less sensitive to training data fluctuations
- **Test MSE** follows a U-shape — optimal λ is at the bottom

This is exactly the bias-variance tradeoff from Chapter 2, now controlled by a single parameter λ.

In [ ]:
# ── Bias-variance decomposition across λ ─────────────────────────────────────
alphas_path = np.logspace(-3, 4, 100)
ridge_cv_path, lasso_cv_path = [], []

for a in alphas_path:
    r = -cross_val_score(Ridge(alpha=a), X_tr_s, y_tr, cv=10,
                          scoring='neg_mean_squared_error').mean()
    ridge_cv_path.append(r)
    try:
        l = -cross_val_score(Lasso(alpha=a, max_iter=10000), X_tr_s, y_tr,
                              cv=10, scoring='neg_mean_squared_error').mean()
    except Exception:
        l = np.nan
    lasso_cv_path.append(l)

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

for ax, path, name, best_a, color in zip(
        axes,
        [ridge_cv_path, lasso_cv_path],
        ['Ridge', 'Lasso'],
        [best_alpha_ridge, best_alpha_lasso],
        ['#e85d20', '#1a7a45']):
    ax.plot(np.log10(alphas_path), path, '-', color=color, lw=2)
    ax.axvline(np.log10(best_a), color='#1e5fa8', lw=2, ls='--',
               label=f'CV optimal λ = {best_a:.4f}')
    ax.set_xlabel('log₁₀(λ)')
    ax.set_ylabel('10-fold CV MSE')
    ax.set_title(f'{name} — CV MSE vs λ\nU-shape = bias-variance tradeoff')
    ax.legend(fontsize=9)

plt.tight_layout()
plt.show()

ols_cv = -cross_val_score(LinearRegression(), X_tr_s, y_tr, cv=10,
                           scoring='neg_mean_squared_error').mean()
print(f"CV MSE summary:")
print(f"  OLS (λ=0):                   {ols_cv:.4f}")
print(f"  Ridge (λ={best_alpha_ridge:.3f}): {min(ridge_cv_path):.4f}")
print(f"  Lasso (λ={best_alpha_lasso:.4f}): {min(lasso_cv_path):.4f}")
print()
print("📌 Left of optimal λ: high variance (overfitting), low bias.")
print("   Right of optimal λ: high bias (underfitting), low variance.")
print("   CV finds the sweet spot automatically.")


## 💼 Exercise

**Task 1 — Elastic Net:** Elastic Net combines Ridge and Lasso: penalty = α·L1 + (1−α)·L2. Import `ElasticNetCV` and fit it on the Hitters training set. What α and λ does CV choose? Compare test MSE to Ridge and Lasso.

**Task 2 — Coefficient stability:** Fit Ridge at λ=0.01, 1, 100, and 10,000. For each, print the coefficient of `CAtBat` (career at-bats). How does it change? At what λ does it become negligible?

**Task 3 — When does Lasso fail?** Simulate X with two perfectly correlated features (r=0.99). Fit Lasso. Which feature does it keep? Run it 5 times with different random seeds — does it always keep the same one? What does this tell you about Lasso's behaviour with correlated features?

In [ ]:
# ── Exercise workspace ───────────────────────────────────────────────────────
# Variables available:
#   X_tr_s, X_te_s, y_tr, y_te, features, scaler
#   ols, ridge_best, lasso_best
#   best_alpha_ridge, best_alpha_lasso

# Task 1 — Elastic Net
from sklearn.linear_model import ElasticNetCV
enet = ElasticNetCV(l1_ratio=[.1,.5,.7,.9,.95,1], cv=10,
                    max_iter=20000, random_state=42).fit(X_tr_s, y_tr)
enet_mse = mean_squared_error(y_te, enet.predict(X_te_s))
print(f"ElasticNet — l1_ratio={enet.l1_ratio_:.2f}  λ={enet.alpha_:.4f}")
print(f"  Non-zero: {np.sum(enet.coef_!=0)}/{len(features)}  Test MSE: {enet_mse:.4f}")
print()

# Task 2 — Coefficient stability
if 'CAtBat' in features:
    idx = features.index('CAtBat')
    print("Ridge coefficient for CAtBat at various λ:")
    for a in [0.01, 1, 100, 10000]:
        coef = Ridge(alpha=a).fit(X_tr_s, y_tr).coef_[idx]
        print(f"  λ={a:>8}: β_CAtBat = {coef:.4f}")

# Task 3 — Lasso with correlated features (your code here)


In [ ]:
# @title 📝 Quiz — Ridge & Lasso Regularisation
# @markdown Answer each question, then run this cell.

# @markdown **Q1:** What does the λ (lambda) parameter control in Ridge and Lasso?
# @markdown **Q2:** Why can Lasso produce exactly-zero coefficients but Ridge cannot?
# @markdown **Q3:** You have 500 features but only 200 observations. Which method would you prefer — Ridge or Lasso? Why?
# @markdown **Q4:** What happens to bias and variance as λ increases from 0 to ∞?
# @markdown **Q5:** When would you choose Elastic Net over either Ridge or Lasso alone?

q1 = "" # @param {type:"string", placeholder:"your answer"}
q2 = "" # @param {type:"string", placeholder:"your answer"}
q3 = "" # @param {type:"string", placeholder:"your answer"}
q4 = "" # @param {type:"string", placeholder:"your answer"}
q5 = "" # @param {type:"string", placeholder:"your answer"}

answers = {"q1": q1, "q2": q2, "q3": q3, "q4": q4, "q5": q5}
missing = [k for k,v in answers.items() if not str(v).strip()]
print(f"  {5-len(missing)}/5 answered — run the submission cell for AI feedback")


In [ ]:
_NB_NAME_ = "Ridge & Lasso Regularisation"
# @title 📋 Quiz Submission
# @markdown **Click ▶ Run** → copy the output → paste into Gemini in this Colab session.

GITHUB_USERNAME = "" # @param {type:"string", placeholder:"your GitHub username e.g. jsmith42"}

_NB_TITLE = globals().get("_NB_NAME_", "this notebook")

if "answers" not in globals():
    print("Run the quiz cell above first, then re-run this cell.")
else:
    qa = "\n".join(
        f"Q{i}: {str(v).strip() or '(no answer)'}"
        for i, (_, v) in enumerate(answers.items(), 1)
    )
    print(f'Please grade my quiz answers for the "{_NB_TITLE}" notebook')
    print(f'from "Data Pattern Recognition for the Rest of Us" (based on ISLP).')
    if GITHUB_USERNAME.strip():
        print(f"Student: @{GITHUB_USERNAME.strip()}")
    print()
    print(qa)
    print()
    print("For each question:")
    print("  1. Say CORRECT, PARTIAL, or INCORRECT")
    print("  2. Explain in 2-3 sentences why")
    print("  3. Give the ideal complete answer")
    print("  4. If wrong/partial, say which Part to revisit")
    print()
    print("End with:")
    print("  - Overall grade: Excellent / Good / Needs Review / Incomplete")
    print("  - A short study plan")


---
### 🐛 Found a bug or have a suggestion?
[Open a GitHub Issue](https://github.com/ladataanalytics/pattern-recognition-notebooks/issues/new?template=bug_report.md)

---
*Data Pattern Recognition for the Rest of Us · [ladataanalytics.github.io/pattern-recognition-notebooks](https://ladataanalytics.github.io/pattern-recognition-notebooks)*